# RAG Evaluation Notebook

Evaluates the hybrid RAG system (vector + BM25 + graph) on regulatory compliance queries.

## Metrics
- **Hit Rate @5**: Proportion of queries where the relevant chunk appears in top-5 results
- **MRR (Mean Reciprocal Rank)**: Average of 1/rank_of_first_relevant across queries
- **NDCG@5**: Normalized Discounted Cumulative Gain at rank 5
- **Faithfulness**: LLM-as-judge check that cited regulations come from retrieved chunks
- **Answer Relevance**: Cosine similarity between answer embedding and question embedding

In [ ]:
import os
import sys

sys.path.insert(0, '..')

os.environ['DJANGO_SETTINGS_MODULE'] = 'config.settings.local'
os.environ['USE_MOCK_BQ'] = 'true'
os.environ['USE_MOCK_GCS'] = 'true'

from compliance_agent.bootstrap import bootstrap_django

bootstrap_django()

In [ ]:
# Test queries with ground-truth relevant documents
TEST_QUERIES = [
    {
        'query': 'When must a PEP alert be escalated to human review?',
        'relevant_docs': ['uiaf_resolucion_285_2007', 'cnbv_cfpior_art95', 'sbs_resolucion_2019_1874'],
        'relevant_articles': ['Art. 14', 'Art. 95', 'Art. 17'],
    },
    {
        'query': '¿Cuál es el umbral de reporte en efectivo para Colombia?',
        'relevant_docs': ['uiaf_resolucion_285_2007', 'uiaf_circular_002_2020'],
        'relevant_articles': ['Art. 7', 'Art. 1'],
    },
    {
        'query': 'Plazos para reportar operaciones sospechosas en México',
        'relevant_docs': ['cnbv_cfpior_art95'],
        'relevant_articles': ['Art. 97'],
    },
    {
        'query': 'Record retention requirements for suspicious transaction reports in Peru',
        'relevant_docs': ['sbs_resolucion_2019_1874'],
        'relevant_articles': ['Art. 23'],
    },
    {
        'query': 'How to classify customers by risk level for AML compliance',
        'relevant_docs': ['uiaf_circular_002_2020', 'sbs_circular_g_2140_2006'],
        'relevant_articles': ['Art. 5', 'Art. 1'],
    },
]

print(f'Test set: {len(TEST_QUERIES)} queries')

In [ ]:
import math


def hit_rate_at_k(results: list[dict], relevant_docs: list[str], k: int = 5) -> float:
    """HR@k = 1 if any relevant doc in top-k, else 0"""
    top_k_refs = [r['document_ref'] for r in results[:k]]
    return 1.0 if any(rel in top_k_refs for rel in relevant_docs) else 0.0

def reciprocal_rank(results: list[dict], relevant_docs: list[str]) -> float:
    """1/rank of first relevant document"""
    for rank, result in enumerate(results, 1):
        if result['document_ref'] in relevant_docs:
            return 1.0 / rank
    return 0.0

def ndcg_at_k(results: list[dict], relevant_docs: list[str], k: int = 5) -> float:
    """NDCG@k = DCG@k / IDCG@k"""
    def dcg(ranked):
        return sum(
            (1.0 if r['document_ref'] in relevant_docs else 0.0) / math.log2(i + 2)
            for i, r in enumerate(ranked[:k])
        )
    ideal = sorted([1.0] * min(len(relevant_docs), k) + [0.0] * k, reverse=True)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal[:k]))
    return dcg(results) / idcg if idcg > 0 else 0.0

print('Metric functions defined.')

In [ ]:
# TODO: Run evaluation (requires DB with indexed documents)
# from compliance_agent.rag import HybridRetriever, BM25Retriever, VectorStoreRetriever, VertexAIEmbedder
#
# embedder = VertexAIEmbedder()
# retriever = HybridRetriever(
#     vector_retriever=VectorStoreRetriever(embedder),
#     bm25_retriever=BM25Retriever(),
# )
#
# results_all = []
# for test in TEST_QUERIES:
#     chunks = await retriever.retrieve(test['query'], top_k=5)
#     retrieved = [{'document_ref': c.document_ref, 'score': c.score} for c in chunks]
#     results_all.append({
#         'query': test['query'],
#         'hr_at_5': hit_rate_at_k(retrieved, test['relevant_docs']),
#         'mrr': reciprocal_rank(retrieved, test['relevant_docs']),
#         'ndcg_at_5': ndcg_at_k(retrieved, test['relevant_docs']),
#     })
#
# import pandas as pd
# df = pd.DataFrame(results_all)
# print(df.mean(numeric_only=True))

print('Evaluation stub — run after indexing regulatory documents.')